# Phase 2 : Nettoyage des données 

In [3]:
import pandas as pd
donnees = pd.read_csv("../data/raw/fr-en-baccalaureat-par-academie.csv",sep=";", keep_default_na=False)

In [4]:
donnees.shape

(43217, 22)

## 1. Suprimer les colonnes dont on a pas besoin 

In [5]:
donnees = donnees.drop(columns=["Code académie", "Nombre d'admis au 1er groupe","Nombre de refusés au 1er groupe","Nombre d'ajournés, passant les épreuves du 2nd groupe","Nombre d'admis à l'issue du 2nd groupe", "Nombre de refusés à l'issue du 2nd groupe","Nombre d'admis avec mention TB avec les félicitations du jury","Nombre d'admis avec mention TB sans les félicitations du jury","Nombre d'admis avec mention B","Nombre d'admis avec mention AB","Nombre d'admis sans mention"])

## Normalisation des noms de colonnes

In [6]:
#mettre en minuscule
donnees.columns = donnees.columns.str.lower()
donnees.columns

Index(['session', 'libellé académie', 'sexe', 'statut du candidat', 'voie',
       'série', 'diplôme spécialité', 'nombre d'inscrits',
       'nombre de présents', 'nombre d'admis totaux',
       'nombre de refusés totaux'],
      dtype='str')

In [7]:
#Remplacer les espaces par des underscores (_)
donnees.columns = donnees.columns.str.replace(" ", "_")
donnees.columns

Index(['session', 'libellé_académie', 'sexe', 'statut_du_candidat', 'voie',
       'série', 'diplôme_spécialité', 'nombre_d'inscrits',
       'nombre_de_présents', 'nombre_d'admis_totaux',
       'nombre_de_refusés_totaux'],
      dtype='str')

In [8]:
#supprimer les apostrophes
donnees.columns = donnees.columns.str.replace("'", "_")

In [9]:
#enlever les accents 
donnees.columns = donnees.columns.str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8")
donnees.columns

Index(['session', 'libelle_academie', 'sexe', 'statut_du_candidat', 'voie',
       'serie', 'diplome_specialite', 'nombre_d_inscrits',
       'nombre_de_presents', 'nombre_d_admis_totaux',
       'nombre_de_refuses_totaux'],
      dtype='str')

## 2. Filtrer les années

In [10]:
# garder que le annes 2024 et 2025
donnees = donnees[donnees["session"].isin([2024, 2025])]
donnees["session"].unique()

array([2024, 2025])

## 3. Filtrer les academies

In [11]:
donnees = donnees[donnees["libelle_academie"].isin(["LILLE", "AMIENS"])]
donnees["libelle_academie"].unique()

<ArrowStringArray>
['AMIENS', 'LILLE']
Length: 2, dtype: str

## 4. Normaliser les noms 

In [12]:
# Passer des MAJ a min
donnees["libelle_academie"] = donnees["libelle_academie"].str.capitalize()
donnees["libelle_academie"].unique()

<ArrowStringArray>
['Amiens', 'Lille']
Length: 2, dtype: str

## 5. Harmonisation 

### 5.1 Colonne sexe 

In [13]:
donnees["sexe"].unique()
donnees["sexe"] = donnees["sexe"].str.replace("FILLES","Féminin")
donnees["sexe"] = donnees["sexe"].str.replace("GARCONS","Masculin")
donnees["sexe"] = donnees["sexe"].str.capitalize()
donnees["sexe"] = donnees["sexe"].str.replace("Feminin","Féminin")
donnees["sexe"].unique()

<ArrowStringArray>
['Féminin', 'Masculin']
Length: 2, dtype: str

### 5.2 Colonne voie

In [14]:
donnees["voie"] = donnees["voie"].str.capitalize()
donnees["voie"].unique()
donnees["voie"] = donnees["voie"].str.replace("Bac professionnel","Professionnel")
donnees["voie"] = donnees["voie"].str.replace("Bac technologique","Technologique")
donnees["voie"] = donnees["voie"].str.replace("Bac general","Général")

### 5.2 Colonne statut candidat

In [15]:
donnees["statut_du_candidat"] = donnees["statut_du_candidat"].str.capitalize()
donnees["statut_du_candidat"] = donnees["statut_du_candidat"].str.replace(" a "," à ")
donnees["statut_du_candidat"].unique()

<ArrowStringArray>
[          'Apprentissage', 'Enseignement à distance',
      'Formation continue',              'Individuel',
                'Scolaire']
Length: 5, dtype: str

## 6. Vérification

### 6.1 Nombre de lignes et colonnes

In [16]:
donnees.shape

(1286, 11)

Après le nettoyage et le filtrage des données sur les années 2024 et 2025 et les académies de Lille et d’Amiens, le jeu de données est passé de 43 217 lignes et 22 colonnes à 1 286 lignes et 11 colonnes.

### 6.2  Vérification de doublons

In [17]:
donnees.duplicated().sum()

np.int64(0)

Aucun doublon intégral n’a été détecté parmi les 1 286 lignes. Aucune suppression supplémentaire n’est donc nécessaire.

## 6.3 Valeurs manquantes 

In [18]:
donnees.isna().sum()

session                     0
libelle_academie            0
sexe                        0
statut_du_candidat          0
voie                        0
serie                       0
diplome_specialite          0
nombre_d_inscrits           0
nombre_de_presents          0
nombre_d_admis_totaux       0
nombre_de_refuses_totaux    0
dtype: int64

Aucune valeur manquante n’a été détectée dans les 11 colonnes conservées.

## Cohérence des effectifs

### Vérification de la cohérence entre les inscrits et les présents

In [19]:
verif = donnees["nombre_de_presents"] > donnees["nombre_d_inscrits"]
verif.value_counts()

False    1286
Name: count, dtype: int64

### Verif entre les admis et les présents

In [20]:
verif2 = donnees["nombre_d_admis_totaux"] > donnees["nombre_de_presents"]
verif2.value_counts()

False    1286
Name: count, dtype: int64

### Somme des candidats admis et refusés correspond au nombre de candidats présents.

In [21]:
candidats = donnees["nombre_d_admis_totaux"] + donnees["nombre_de_refuses_totaux"]
verif3 = candidats == donnees["nombre_de_presents"]
verif3.value_counts()

True    1286
Name: count, dtype: int64

### Conclusion — Cohérence des effectifs
Les vérifications réalisées sur les 1 286 lignes montrent que :
- Le nombre de présents ne dépasse jamais le nombre d'inscrits.
- Le nombre d'admis ne dépasse jamais le nombre de présents.
- La somme des admis et des refusés correspond toujours au nombre de présents.

# Phase 3 préparation à l'importation dans PostgreSQL 

## 1.Vérification des noms de colonnes

In [22]:
donnees.columns

Index(['session', 'libelle_academie', 'sexe', 'statut_du_candidat', 'voie',
       'serie', 'diplome_specialite', 'nombre_d_inscrits',
       'nombre_de_presents', 'nombre_d_admis_totaux',
       'nombre_de_refuses_totaux'],
      dtype='str')

# 2. Harmonisation des noms de colonnes avec PostgreSQL

In [24]:
donnees = donnees.rename(columns={
      "statut_du_candidat" : "statut_candidat" , 
      "nombre_d_inscrits" : "nb_inscrits" ,    
      "nombre_de_presents" :"nb_presents" ,
      "nombre_d_admis_totaux":"nb_admis_totaux",
      "nombre_de_refuses_totaux": "nb_refuses_totaux"
      })                                                                                                                                                                                                                                

In [25]:
donnees.columns

Index(['session', 'libelle_academie', 'sexe', 'statut_candidat', 'voie',
       'serie', 'diplome_specialite', 'nb_inscrits', 'nb_presents',
       'nb_admis_totaux', 'nb_refuses_totaux'],
      dtype='str')

## 3. Vérification des types de données


In [26]:
donnees.dtypes

session               int64
libelle_academie        str
sexe                    str
statut_candidat         str
voie                    str
serie                   str
diplome_specialite      str
nb_inscrits           int64
nb_presents           int64
nb_admis_totaux       int64
nb_refuses_totaux     int64
dtype: object

Les 11 colonnes du DataFrame possèdent des types de données
compatibles avec ceux attendus dans la table resultats_bac.
Aucune conversion de type n'est nécessaire à ce stade.

## 4. Contrôle complémentaire des chaînes vides

In [27]:
(donnees["diplome_specialite"].str.strip()=="").sum()

np.int64(0)

In [29]:
donnees.select_dtypes(include="str").apply( lambda colonne: (colonne.str.strip() == "").sum())

libelle_academie      0
sexe                  0
statut_candidat       0
voie                  0
serie                 0
diplome_specialite    0
dtype: int64

Le contrôle complémentaire des chaînes vides a été réalisé
sur les six colonnes textuelles du DataFrame.

Aucune chaîne vide ou composée uniquement d'espaces
n'a été détectée.

Aucune correction supplémentaire n'est nécessaire.

In [30]:
donnees.select_dtypes(include="int64").max()

session               2025
nb_inscrits          12402
nb_presents          12402
nb_admis_totaux      12077
nb_refuses_totaux      370
dtype: int64

In [31]:
donnees.select_dtypes(include="int64").min()

session              2024
nb_inscrits             1
nb_presents             0
nb_admis_totaux         0
nb_refuses_totaux       0
dtype: int64

## 5. Conclusion de la préparation des données

Les 11 noms de colonnes correspondent à la structure
de la table resultats_bac dans PostgreSQL.

Les types de données sont compatibles et les valeurs
numériques respectent les limites du type INTEGER.

Aucune chaîne vide n'a été détectée.

Les 1 286 lignes sont prêtes pour l'étape d'importation
dans PostgreSQL.

# Phase 4 : Connexion à PostgreSQL

## 1. Importation des bibliothèques

In [32]:
from sqlalchemy import create_engine

# Phase 4 : Connexion et importation dans PostgreSQL

## 1. Configuration de la connexion

La bibliothèque SQLAlchemy permet de gérer la connexion
à PostgreSQL en utilisant le pilote Psycopg.

Le mot de passe est demandé lors de l'exécution afin
de ne pas stocker d'identifiants confidentiels dans Git.

In [34]:
from sqlalchemy import create_engine, URL, text
from getpass import getpass

# Configuration de la connexion PostgreSQL
url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=getpass("Mot de passe PostgreSQL : "),
    host="localhost",
    port=5432,
    database="parcoursup"
)

# Création du moteur de connexion
engine = create_engine(url)

# Test de la connexion à la base de données
with engine.connect() as connexion:
    resultat = connexion.execute(text("SELECT current_database()"))
    print("Connexion réussie :", resultat.scalar())

Connexion réussie : parcoursup


## 2. Importation des données du baccalauréat

Les données nettoyées et harmonisées sont importées
dans la table resultats_bac du schéma public.

La méthode to_sql() de Pandas permet de transférer
les données du DataFrame vers PostgreSQL.

L'importation conserve la structure SQL existante.
Un contrôle préalable permet d'éviter une double
importation des données.

In [36]:
# Importation des données dans PostgreSQL
with engine.begin() as connexion:

    # Vérification du contenu de la table
    nombre_lignes = connexion.execute(
        text("SELECT COUNT(*) FROM public.resultats_bac")
    ).scalar()

    # Importation uniquement si la table est vide
    if nombre_lignes == 0:
        donnees.to_sql(
            name="resultats_bac",
            con=connexion,
            schema="public",
            if_exists="append",
            index=False
        )

        print(f"Importation réussie : {len(donnees)} lignes")

    else:
        print("Importation annulée : la table contient déjà des données.")

Importation annulée : la table contient déjà des données.


In [37]:
donnees.groupby(["session", "libelle_academie"]).size()

session  libelle_academie
2024     Amiens              281
         Lille               367
2025     Amiens              297
         Lille               341
dtype: int64

## 3. Validation de l'importation dans PostgreSQL

L'importation des données du baccalauréat dans la table `public.resultats_bac` a été contrôlée depuis DBeaver.

Les vérifications réalisées confirment :

- La présence des 1 286 lignes attendues.
- La présence des sessions 2024 et 2025.
- La présence des académies de Lille et d'Amiens.
- Une répartition identique des lignes par session et par académie entre le DataFrame Python et PostgreSQL.
- Le fonctionnement du contrôle empêchant une nouvelle importation lorsque la table contient déjà des données.

Les contrôles de volumétrie et de répartition sont donc validés. Ils ne constituent toutefois pas une comparaison exhaustive de chaque valeur importée.

Les données Parcoursup 2024 et 2025 seront importées ultérieurement, après finalisation de leur nettoyage.